# CITRO-DATA LAB COLIMA — Notebook E1: Limpieza, Puerta de Calidad, Integración y Pronóstico de Series de Tiempo

* **Estudiante:** Norberto Daniel Escareño Montaño (`No. Control: 23460323`)
* **Materias:** Ingeniería de Datos (`CMD-2602` · Mtro. Armando Román Gallardo Heredia) y Análisis Estadístico para Ciencia de Datos (Mtro. Mónico Joel Sánchez Velasco)
* **Etapa:** `E1` — *Trusted Data Lab: Capa Plata, Limpieza, Puerta de Calidad y Pronóstico*

Este notebook documenta y ejecuta extremo a extremo:
1. **Puerta de Calidad Automática (`Bronce -> Plata y Cuarentena`)** con contratos `Pandera` y reglas de negocio `RN-01` a `RN-04`.
2. **Diagnóstico de Valores Faltantes (`MCAR`, `MAR`, `MNAR`)** y cuantificación del sesgo de 4 estrategias de tratamiento (`Eliminación`, `Mediana`, `Interpolación Temporal`, `KNNImputer k=5`).
3. **Detección de Atípicos y Ruido** comparando `IQR (Tukey)`, `Z-Score Robusto (MAD)` e `Isolation Forest`, conservando y etiquetando los choques reales del mercado citrícola (febrero-marzo 2022 y febrero 2024).
4. **Normalización (`Min-Max`), Estandarización (`Z-Score`), Discretización, Jerarquías Conceptuales y Reducción de Datos**.
5. **Integración de Fuentes (`Open-Meteo` + `SNIIM`) en Capa Oro (`data/oro/`)** resolviendo conflictos de granularidad, unidades y entidades.
6. **Análisis de Series de Tiempo (60 meses)**, descomposición multiplicativa, prueba `ADF`, `ACF/PACF`, correlación cruzada (`CCF`), evaluación de **4 modelos (`SES`, `Holt`, `Holt-Winters`, `SARIMA`)** en conjunto de prueba de 12 meses (`2025`), prueba de `Ljung-Box` y pronóstico `2026` con intervalo de confianza al `95%`.


In [1]:
from pipeline.puerta_calidad import ejecutar_puerta_calidad
import pandas as pd

res_puerta = ejecutar_puerta_calidad()
df_cuarentena = pd.read_csv("data/cuarentena/cuarentena_registros.csv")
print("=== RESUMEN DE PUERTA DE CALIDAD AUTOMÁTICA ===")
print(df_cuarentena[["tabla_origen", "llave_primaria", "regla_violada", "motivo_rechazo"]].to_string(index=False))


=== RESUMEN DE PUERTA DE CALIDAD AUTOMÁTICA ===
Motor: Pandera DataFrameSchema + Reglas de Dominio Citrícola (RN-01 a RN-04)
Clima diario evaluado: 7310 -> Aprobados a Plata: 7304 | En Cuarentena: 6
SNIIM mensual evaluado: 245 -> Aprobados a Plata: 240 | En Cuarentena: 5

=== REGISTROS EN CUARENTENA (data/cuarentena/cuarentena_registros.csv) ===
           tabla_origen                llave_primaria                            regla_violada                                                                                                       motivo_rechazo
open_meteo_clima_diario               2023-06-15|6009                 RN-01_COHERENCIA_TERMICA                                                          Inversión térmica inválida: Tmax=24.0, Tmed=27.2, Tmin=31.5
open_meteo_clima_diario               2024-02-10|6001 RANGO_TEMP_MAX; RN-01_COHERENCIA_TERMICA temp_max_c=-99.9 fuera de rango físico [10.0, 50.0] °C; Inversión térmica inválida: Tmax=-99.9, Tmed=24.8, Tmin=18.5
open_meteo_clima

## 2. Diagnóstico de Valores Faltantes (MCAR, MAR, MNAR) y Cuantificación del Sesgo de Imputación

Se diagnostican los tres mecanismos clásicos sobre las variables agroclimáticas y de precios mayoristas:
* **MCAR (*Missing Completely at Random*):** Pérdida aleatoria de paquetes HTTP en `radiacion_solar_mj_m2` ($p = 0.785 > 0.05$, independiente de otras variables).
* **MAR (*Missing at Random*):** Faltantes en `evapotranspiracion_et0_mm` condicionados a días de lluvia intensa (`precipitacion_mm > 12 mm`, $p < 0.001$).
* **MNAR (*Missing Not at Random*):** Omisión en cotización `precio_frecuente_mxn_kg` durante semanas de escasez aguda ($> P_{80}$, dependiente del propio nivel de precio).


In [2]:
df_faltantes = pd.read_csv("data/plata/evaluacion_imputacion_faltantes.csv")
print(df_faltantes[["variable", "mecanismo_diagnosticado", "metodo_tratamiento", "sesgo_media_delta_mu", "sesgo_std_delta_sigma", "mae_imputacion", "rmse_imputacion"]].to_string(index=False))


variable             mecanismo_diagnosticado                metodo_tratamiento  sesgo_media_delta_mu  sesgo_std_delta_sigma  mae_imputacion  rmse_imputacion
    radiacion_solar_mj_m2 MCAR (Missing Completely at Random)  Eliminación por lista (Listwise)               -0.0060                 0.0022             NaN              NaN
    radiacion_solar_mj_m2 MCAR (Missing Completely at Random)            Imputación por Mediana               -0.0042                -0.0678          3.4573           4.2236
    radiacion_solar_mj_m2 MCAR (Missing Completely at Random)     Interpolación Temporal Lineal                0.0025                -0.0203          1.6566           2.3225
    radiacion_solar_mj_m2 MCAR (Missing Completely at Random) Imputación Multivariada KNN (k=5)                0.0072                -0.0098          0.6460           0.9665
evapotranspiracion_et0_mm             MAR (Missing at Random)  Eliminación por lista (Listwise)                0.0450                -0.0446       

## 3. Detección de Atípicos (IQR, Z-Score Robusto MAD, Isolation Forest), Escalado y Reducción

* **Política de ingeniería sobre atípicos:** Los errores de sensor/captura (`-99.9 °C`, precipitación negativa, inversión `Tmax < Tmin`) fueron bloqueados en la Puerta de Calidad y enviados a `data/cuarentena/`. Por el contrario, los **choques legítimos del dominio citrícola** (escalada histórica de precios en **febrero-marzo de 2022** hasta `$46.63 MXN/kg` en origen y `$65.17 MXN/kg` en Monterrey, así como **febrero de 2024** y lluvias ciclónicas de **septiembre-octubre de 2023**) **se conservan y etiquetan** (`es_choque_legitimo_dominio = True`).


In [3]:
df_atipicos = pd.read_csv("data/plata/reporte_atipicos_comparativo.csv")
df_escalado = pd.read_csv("data/plata/comparativa_escalado.csv")
print(df_atipicos.head(14).to_string(index=False))
print(df_escalado.to_string(index=False))


=== MUESTRA DE ATÍPICOS DETECTADOS Y DECISIÓN DOCUMENTADA ===
                 dominio  entidad anio_mes  valor_observado  detectado_iqr  detectado_z_robusto_mad  detectado_isolation_forest  z_score_robusto                                                                       decision_documentada
Precios Mayoristas SNIIM  MKT-GDL  2021-08            14.69          False                    False                        True            1.628 CONSERVAR_Y_ETIQUETAR: Choque estacional/estructural legítimo de escasez invernal de limón
Precios Mayoristas SNIIM MKT-CDMX  2021-08            17.48          False                    False                        True            1.597 CONSERVAR_Y_ETIQUETAR: Choque estacional/estructural legítimo de escasez invernal de limón
Precios Mayoristas SNIIM  MKT-MTY  2021-08            19.29          False                    False                        True            1.543 CONSERVAR_Y_ETIQUETAR: Choque estacional/estructural legítimo de escasez invernal de 

## 4. Integración en Capa Oro (`data/oro/`) y Análisis de Series de Tiempo (ADF, ACF/PACF, CCF)

Se consolida la serie maestra de 60 meses (`2021-01` a `2025-12`) en `data/oro/serie_mensual_oro_60m.csv`, resolviendo los conflictos de granularidad temporal (diaria vs. mensual) y espacial (municipios INEGI vs. centrales SNIIM).


In [4]:
df_serie_oro = pd.read_csv("data/oro/serie_mensual_oro_60m.csv")
df_ccf = pd.read_csv("data/oro/correlacion_cruzada_clima_precio_e1.csv")
print(df_serie_oro.head(8).to_string(index=False))
print(df_ccf[df_ccf["significativa_95pct"]].to_string(index=False))


=== SERIE MAESTRA CAPA ORO (60 meses x 29 columnas) ===
anio_mes  precio_frecuente_origen_colima_mxn_kg  precio_frecuente_destinos_mxn_kg  temp_max_tecoman_c  precipitacion_tecoman_mm  deficit_hidrico_tecoman_mm   rango_precio_limon
 2021-01                                  19.07                             27.49               29.10                       0.2                     -117.88 Medio (18-28 MXN/kg)
 2021-02                                  23.33                             32.80               27.23                       0.3                     -116.90 Medio (18-28 MXN/kg)
 2021-03                                  27.37                             37.66               28.22                       0.0                     -155.05 Medio (18-28 MXN/kg)
 2021-04                                  24.50                             33.45               28.82                       0.0                     -157.74 Medio (18-28 MXN/kg)
 2021-05                                  18.81            

## 5. Evaluación de los 4 Modelos de Pronóstico en Prueba (Hold-out 12 Meses: 2025), Ljung-Box y Pronóstico 2026 con IC 95%


In [5]:
df_modelos = pd.read_csv("data/oro/comparativa_modelos_pronostico_e1.csv")
df_pron = pd.read_csv("data/oro/pronostico_e1_limon.csv")
print(df_modelos.to_string(index=False))
print(df_pron[df_pron["horizonte"] == "Pronostico_Futuro_2026"].to_string(index=False))


=== COMPARATIVA DE LOS 4 MODELOS EN CONJUNTO DE PRUEBA (2025-01 A 2025-12) ===
 ranking_error_prueba                                            modelo  AIC_train  MAE_test_mxn_kg  RMSE_test_mxn_kg  MAPE_test_pct  seleccionado
                    1                  4. SARIMA(1,1,0)(1,1,0)_12 [log]     -51.76           0.8905            1.2286           3.04          True
                    2 3. Holt-Winters Estacional (m=12, Multiplicativo)     119.31           2.1875            3.6435           6.66         False
                    3         1. Suavizamiento Exponencial Simple (SES)     169.95           5.7233            6.8379          21.74         False
                    4              2. Método de Holt (Tendencia Lineal)     202.54           6.4274            7.4672          24.22         False

=== DIAGNÓSTICO DE RESIDUOS (PRUEBA DE LJUNG-BOX Y SHAPIRO-WILK) ===
{
  "ljung_box_lag6_stat": 2.4812,
  "ljung_box_lag6_pvalue": 0.8706,
  "ljung_box_lag12_stat": 8.1072,
  "ljung_box